# Airbnb Data Cleaning Pipeline

This notebook cleans the raw Airbnb Open Data CSV using PySpark. It covers column renaming, deduplication, type casting, date parsing, value validation, missing value imputation, feature engineering, and writing the final output.

## Setup

We start a Spark session and define the input and output file paths. `inferSchema=True` lets Spark guess column types on load, though many of these will be corrected later due to mixed formatting in the raw file.

In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, regexp_replace, when, to_date, year, month
from pyspark.sql.types import DoubleType
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("AirbnbCleaning").getOrCreate()

INPUT_FILE = "Airbnb_Open_Data.csv"
OUTPUT_FILE = "airbnb_cleaned"

## Load Raw Data

We read the CSV and print its shape and column names. The raw dataset has 102,998 rows and 26 columns. Column names are inconsistent at this point, with mixed casing and spaces, which we address in the next step.

In [32]:
df = spark.read.csv(INPUT_FILE, header=True, inferSchema=True)

print("Raw shape:", (df.count(), len(df.columns)))
print("Original columns:", df.columns)

Raw shape: (102998, 26)
Original columns: ['id', 'NAME', 'host id', 'host_identity_verified', 'host name', 'neighbourhood group', 'neighbourhood', 'lat', 'long', 'country', 'country code', 'instant_bookable', 'cancellation_policy', 'room type', 'Construction year', 'price', 'service fee', 'minimum nights', 'number of reviews', 'last review', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365', 'house_rules', 'license']


## Standardize Column Names

All column names are converted to lowercase with spaces and slashes replaced by underscores. This makes them consistent and easier to reference programmatically throughout the rest of the pipeline.

In [33]:
for col_name in df.columns:
    new_name = col_name.strip().lower().replace(" ", "_").replace("/", "_")
    df = df.withColumnRenamed(col_name, new_name)

## Remove Duplicate Rows

We first drop fully duplicate rows, then deduplicate again on the `id` column to catch cases where the same listing appears multiple times with slightly different values in other fields.

In [34]:
df = df.dropDuplicates()

if "id" in df.columns:
    df = df.dropDuplicates(["id"])

## Clean String Columns

For every string column, we trim leading and trailing whitespace and replace common placeholder strings like empty strings, `"nan"`, and `"None"` with actual null values. This makes downstream null handling consistent.

In [35]:
for c, t in df.dtypes:
    if t == "string":
        df = df.withColumn(c, trim(col(c)))
        df = df.withColumn(
            c,
            when(col(c).isin("", "nan", "None"), None).otherwise(col(c))
        )

## Define a Safe Numeric Cast Helper

The `safe_cast` function strips out any non-numeric characters from a column (such as dollar signs or commas), then attempts to cast the result to a double. Values that reduce to meaningless strings like `"."` or `"-"` after stripping are returned as null rather than causing a parse error.

In [36]:
from pyspark.sql.functions import col, regexp_replace, when, trim

def safe_cast(col_name):
    cleaned = regexp_replace(col(col_name).cast("string"), r"[^0-9.-]", "")
    cleaned = trim(cleaned)

    return when(
        (cleaned == "") |
        (cleaned == ".") |
        (cleaned == "-") |
        (cleaned == "-.") |
        (cleaned == ".-"),
        None
    ).when(
        cleaned.rlike(r"^-?\d+(\.\d+)?$"),
        cleaned.cast("double")
    ).otherwise(None)

## Cast Price Columns

The `price` and `service_fee` columns often contain currency symbols and formatting characters in the raw data. We apply `safe_cast` to strip those out and convert both columns to numeric doubles.

In [37]:
for c in ["price", "service_fee"]:
    if c in df.columns:
        df = df.withColumn(c, safe_cast(c))

## Cast Remaining Numeric Columns

We apply the same `safe_cast` function to all other columns that should be numeric, including coordinates, review counts, availability, and construction year. This ensures type consistency across the dataset before any filtering or imputation.

In [38]:
numeric_cols = [
    "lat", "long", "construction_year", "minimum_nights",
    "number_of_reviews", "reviews_per_month",
    "review_rate_number", "calculated_host_listings_count",
    "availability_365"
]

for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(c, safe_cast(c))

## Parse the Last Review Date

The `last_review` column comes in multiple date formats across rows. We try three format patterns in order using `coalesce`, which returns the first non-null result. The final column is cast to a proper date type.

In [39]:
from pyspark.sql.functions import col, coalesce, to_date, try_to_timestamp, lit

if "last_review" in df.columns:
    df = df.withColumn(
        "last_review",
        to_date(
            coalesce(
                try_to_timestamp(col("last_review"), lit("M/d/yyyy")),
                try_to_timestamp(col("last_review"), lit("MM/dd/yyyy")),
                try_to_timestamp(col("last_review"), lit("yyyy-MM-dd"))
            )
        )
    )

## Fix Categorical Values

The `neighbourhood_group` column contains known typos from the raw data. We correct "manhatan" to "Manhattan" and "brookln" to "Brooklyn". We also normalize the `instant_bookable` column to lowercase so values like "True" and "true" are treated identically.

In [40]:
if "neighbourhood_group" in df.columns:
    df = df.replace(
        {"manhatan": "Manhattan", "brookln": "Brooklyn"},
        subset=["neighbourhood_group"]
    )

if "instant_bookable" in df.columns:
    df = df.withColumn("instant_bookable", lower(col("instant_bookable")))

## Validate Numeric Ranges

We clip values that fall outside of logically valid ranges. A listing cannot have fewer than 1 minimum night, availability must be between 0 and 365, review ratings must be between 1 and 5, and construction years must fall within a sensible historical window. Out-of-range values are set to null.

In [41]:
if "minimum_nights" in df.columns:
    df = df.withColumn(
        "minimum_nights",
        when(col("minimum_nights") < 1, None).otherwise(col("minimum_nights"))
    )

if "availability_365" in df.columns:
    df = df.withColumn(
        "availability_365",
        when((col("availability_365") < 0) | (col("availability_365") > 365), None)
        .otherwise(col("availability_365"))
    )

if "review_rate_number" in df.columns:
    df = df.withColumn(
        "review_rate_number",
        when((col("review_rate_number") < 1) | (col("review_rate_number") > 5), None)
        .otherwise(col("review_rate_number"))
    )

if "construction_year" in df.columns:
    df = df.withColumn(
        "construction_year",
        when((col("construction_year") < 1900) | (col("construction_year") > 2026), None)
        .otherwise(col("construction_year"))
    )

## Impute Missing Values

Rows where `price` is null are dropped entirely since price is the target variable for downstream modeling. For categorical columns, nulls are filled with the most frequent value (mode). For numeric columns, nulls are filled with the median, computed using approximate quantiles to keep this efficient at scale.

In [42]:
if "price" in df.columns:
    df = df.dropna(subset=["price"])

categorical_cols = [
    "neighbourhood_group", "room_type", "cancellation_policy",
    "host_identity_verified", "instant_bookable"
]

for c in categorical_cols:
    if c in df.columns:
        mode_val = df.groupBy(c).count().orderBy(col("count").desc()).first()
        if mode_val:
            df = df.fillna({c: mode_val[0]})

numeric_fill_cols = [
    "service_fee", "minimum_nights", "reviews_per_month",
    "review_rate_number", "calculated_host_listings_count",
    "availability_365"
]

for c in numeric_fill_cols:
    if c in df.columns:
        quantiles = df.approxQuantile(c, [0.5], 0.01)
        if quantiles:
            df = df.fillna({c: quantiles[0]})

## Feature Engineering

We derive three new columns from existing data. The year and month are extracted from `last_review` to enable time-based analysis. A `host_type` label is assigned based on listing count, with hosts managing 3 or more listings classified as professional. Finally, `price_category` splits listings into Low, Medium, and High tiers using the 33rd and 66th percentiles as thresholds.

In [43]:
from pyspark.sql.functions import year, month, col, when

if "last_review" in df.columns:
    df = df.withColumn("review_year", year(col("last_review")))
    df = df.withColumn("review_month", month(col("last_review")))

if "calculated_host_listings_count" in df.columns:
    df = df.withColumn(
        "host_type",
        when(col("calculated_host_listings_count") >= 3, "professional")
        .otherwise("casual")
    )

if "price" in df.columns:
    quantiles = df.approxQuantile("price", [0.33, 0.66], 0.01)
    if len(quantiles) == 2:
        low, mid = quantiles
        df = df.withColumn(
            "price_category",
            when(col("price") <= low, "Low")
            .when(col("price") <= mid, "Medium")
            .otherwise("High")
        )

## Final Cleanup and Drop Unused Columns

We do a final pass on `minimum_nights` to null out any values above 365 (clearly erroneous) and re-impute with the median. The `license` and `house_rules` columns are dropped as they are mostly empty and not useful for analysis.

In [44]:
if "minimum_nights" in df.columns:
    quantiles = df.approxQuantile("minimum_nights", [0.5], 0.01)
    if quantiles:
        df = df.withColumn(
            "minimum_nights",
            when(col("minimum_nights") > 365, None).otherwise(col("minimum_nights"))
        ).fillna({"minimum_nights": quantiles[0]})

for col_to_drop in ["license", "house_rules"]:
    if col_to_drop in df.columns:
        df = df.drop(col_to_drop)

## Write Output

We write the cleaned DataFrame to a single CSV file using `coalesce(1)`, which merges all Spark partitions into one output file. The final dataset has 101,661 rows and 28 columns, reflecting the rows dropped due to missing prices and the new columns added during feature engineering.

In [45]:
OUTPUT_FILE = "airbnb_cleaned_single"

df.coalesce(1).write.mode("overwrite").option("header", True).csv(OUTPUT_FILE)

print("Cleaned shape:", (df.count(), len(df.columns)))

[Stage 194:>                                                        (0 + 1) / 1]

Cleaned shape: (101661, 28)
